In [1]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)


Loading model on: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
import pandas as pd

intent_df = pd.read_csv("amazon_intent_base.csv")

print("Rows:", len(intent_df))
print("Columns:", intent_df.columns.tolist())

Rows: 99184
Columns: ['customer_tweet_id', 'customer_id', 'customer_query', 'clean_customer_query', 'amazon_tweet_id', 'amazon_response']


In [3]:
# discovering only the customer intents and not the amazon replies yet  because we want to classify what the customer wnat to say
queries = (
    intent_df["clean_customer_query"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("Number of queries:", len(queries))
print("First 5 queries:")

for query in queries[:5]:
    print("-", query)

Number of queries: 99184
First 5 queries:
- 電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんでしょうね。
- こちらこそありがとうございました。
- 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day
- Okay, danke für die Info
- Yeah this is crazy we’re less than a week away and still no Shipping Information on something that we Pre-ordered back in August


In [4]:
# embedding the data points
embeddings = model.encode(
    queries,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/3100 [00:00<?, ?it/s]

Embedding shape: (99184, 384)


In [51]:
import pandas as pd

# Load dataset
intent_df = pd.read_csv("amazon_intent_base.csv")

# Random 100 queries
sample_100 = (
    intent_df[
        ["customer_tweet_id", "clean_customer_query"]
    ]
    .dropna(subset=["clean_customer_query"])
    .sample(n=100, random_state=42)
    .reset_index(drop=True)
)

print("Selected:", len(sample_100))
print("\nFirst 10:")
print(sample_100.head(10).to_string(index=False))

Selected: 100

First 10:
 customer_tweet_id                                                                                                                                                                                          clean_customer_query
          594532.0                                                                                                                                                                                                           UPS
          588782.0 I just got a package (a birthday gift for tomorrow) and the packaging and the item itself smell very strongly of smoke. I'm assuming the driver at Intelcom was smoking in the car and now my gift is ruined.
         1704079.0                                                                                             Once again, I'm not getting my packages on time from #Intelcom. So much for 2 day delivery! This is the 5th time!
          831799.0                                                         

In [52]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen3:4b"

def ask_ollama_json(prompt):

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "format": "json",
        "options": {
            "temperature": 0,
            "num_predict": 800
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    data = response.json()

    return json.loads(data["response"])

In [53]:
import time

# Start with no intents
existing_intents = []

all_labels = []

BATCH_SIZE = 10
MAX_INTENTS = 15

for start in range(0, len(sample_100), BATCH_SIZE):

    batch = sample_100.iloc[start:start + BATCH_SIZE]

    queries_for_llm = []

    for _, row in batch.iterrows():
        queries_for_llm.append({
            "id": str(row["customer_tweet_id"]),
            "query": row["clean_customer_query"]
        })

    prompt = f"""
You are designing a compact intent taxonomy for Amazon customer support.

Your job is to classify each customer query.

CURRENT INTENTS:
{json.dumps(existing_intents, indent=2)}

IMPORTANT RULES:

1. ALWAYS reuse an existing intent when it adequately describes
   the customer's underlying problem.

2. Do NOT create a new intent just because the wording is different.

3. Do NOT create separate intents based on:
   - emotion
   - country
   - product name
   - company/brand
   - wording
   - minor variations of the same problem

4. Create a NEW intent only when none of the existing intents
   adequately represents the underlying customer problem.

5. Prefer broader reusable intents over narrow intents.

6. Keep the taxonomy as small as reasonably possible.

7. Do not exceed {MAX_INTENTS} total intents.

8. "Non-Informative" should be used for messages that do not
   contain a meaningful customer-support request or problem,
   such as simple thanks, acknowledgements, or vague responses.

9. If the query is genuinely unclear and cannot reasonably fit
   an existing intent, use "Other".

10. Intent names must be short and reusable.
    Use 2-4 words where possible.

CURRENT INTENTS:
{json.dumps(existing_intents, indent=2)}

CUSTOMER QUERIES:
{json.dumps(queries_for_llm, indent=2)}

For every query, select an existing intent or create a new one
only when necessary.

Return ONLY this JSON structure:

{{
    "intents": [
        {{
            "name": "intent name",
            "definition": "short definition"
        }}
    ],
    "labels": [
        {{
            "id": "tweet id",
            "intent": "intent name",
            "confidence": 0.0
        }}
    ]
}}

The "intents" field must contain the COMPLETE updated intent
taxonomy, including existing intents and any genuinely necessary
new intents.

The "labels" field must contain exactly one label for every query.
"""

    print(f"\nProcessing queries {start + 1}-{min(start + BATCH_SIZE, len(sample_100))}...")

    try:
        result = ask_ollama_json(prompt)

        # Update global taxonomy
        existing_intents = result.get("intents", [])

        # Store labels
        all_labels.extend(result.get("labels", []))

        print("Current number of intents:", len(existing_intents))

        for intent in existing_intents:
            print(" -", intent["name"])

    except Exception as e:
        print("ERROR:", e)

    time.sleep(0.5)


Processing queries 1-10...
Current number of intents: 1
 - Non-Informative

Processing queries 11-20...
Current number of intents: 1
 - Non-Informative

Processing queries 21-30...
Current number of intents: 1
 - Non-Informative

Processing queries 31-40...
Current number of intents: 1
 - Non-Informative

Processing queries 41-50...
Current number of intents: 1
 - Non-Informative

Processing queries 51-60...
Current number of intents: 4
 - Non-Informative
 - DeliveryDelay
 - FormFilling
 - OrderStatus

Processing queries 61-70...
Current number of intents: 4
 - Non-Informative
 - DeliveryDelay
 - FormFilling
 - OrderStatus

Processing queries 71-80...
Current number of intents: 4
 - Non-Informative
 - DeliveryDelay
 - FormFilling
 - OrderStatus

Processing queries 81-90...
Current number of intents: 4
 - Non-Informative
 - DeliveryDelay
 - FormFilling
 - OrderStatus

Processing queries 91-100...
Current number of intents: 4
 - Non-Informative
 - DeliveryDelay
 - FormFilling
 - OrderSt

In [56]:
print("FINAL TAXONOMY")
print("=" * 50)

for i, intent in enumerate(existing_intents, 1):
    print(f"\n{i}. {intent['name']}")
    print(f"   {intent['definition']}")

FINAL TAXONOMY

1. Non-Informative
   Messages that do not contain a meaningful customer-support request or problem

2. DeliveryDelay
   Customer reports delays in package delivery

3. FormFilling
   Customer complains about excessive form filling

4. OrderStatus
   Customer asks about order status or delivery progress


In [57]:
labels_df = pd.DataFrame(all_labels)

labeled_100 = sample_100.merge(
    labels_df,
    left_on="customer_tweet_id",
    right_on="id",
    how="left"
)

labeled_100.drop(columns=["id"], inplace=True)

print("Labeled rows:", len(labeled_100))
print("\nLabel distribution:")
print(labeled_100["intent"].value_counts())

ValueError: You are trying to merge on float64 and object columns for key 'customer_tweet_id'. If you wish to proceed you should use pd.concat